In [1]:
import pandas as pd

main = pd.read_excel('../data/raw/ethiopia_fi_unified_data.xlsx', sheet_name='ethiopia_fi_unified_data')
impact = pd.read_excel('../data/raw/ethiopia_fi_unified_data.xlsx', sheet_name='Impact_sheet')
ref = pd.read_excel('../data/raw/reference_codes.xlsx', sheet_name='reference_codes')

print(main.shape, impact.shape, ref.shape)
main.head()

(43, 34) (14, 35) (71, 4)


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN


In [2]:
print("Record types:\n", main['record_type'].value_counts(), "\n")
print("Pillars:\n", main['pillar'].value_counts(dropna=False), "\n")
print("Event categories:\n", main['category'].value_counts(dropna=False), "\n")
print("Source types:\n", main['source_type'].value_counts(dropna=False), "\n")
print("Confidence:\n", main['confidence'].value_counts(dropna=False))

Record types:
 record_type
observation    30
event          10
target          3
Name: count, dtype: int64 

Pillars:
 pillar
ACCESS           16
USAGE            11
NaN              10
GENDER            5
AFFORDABILITY     1
Name: count, dtype: int64 

Event categories:
 category
NaN               33
product_launch     2
infrastructure     2
policy             2
market_entry       1
milestone          1
partnership        1
pricing            1
Name: count, dtype: int64 

Source types:
 source_type
operator      15
survey        10
regulator      7
research       4
policy         3
calculated     2
news           2
Name: count, dtype: int64 

Confidence:
 confidence
high      40
medium     3
Name: count, dtype: int64


In [3]:
obs = main[main['record_type'] == 'observation']
print("Date range:", obs['observation_date'].min(), "to", obs['observation_date'].max())
print("\nIndicator coverage:\n", obs['indicator_code'].value_counts())

Date range: 2014-12-31 00:00:00 to 2025-12-31 00:00:00

Indicator coverage:
 indicator_code
ACC_OWNERSHIP         6
ACC_FAYDA             3
ACC_MM_ACCOUNT        2
ACC_4G_COV            2
USG_P2P_COUNT         2
GEN_GAP_ACC           2
ACC_MOBILE_PEN        1
USG_ATM_COUNT         1
USG_ATM_VALUE         1
USG_CROSSOVER         1
USG_P2P_VALUE         1
USG_TELEBIRR_USERS    1
USG_TELEBIRR_VALUE    1
USG_MPESA_ACTIVE      1
USG_MPESA_USERS       1
USG_ACTIVE_RATE       1
AFF_DATA_INCOME       1
GEN_MM_SHARE          1
GEN_GAP_MOBILE        1
Name: count, dtype: int64


In [4]:
events = main[main['record_type'] == 'event'][['record_id','indicator','category','observation_date']]
print(events.to_string(index=False))

record_id                              indicator       category observation_date
 EVT_0001                        Telebirr Launch product_launch       2021-05-17
 EVT_0002   Safaricom Ethiopia Commercial Launch   market_entry       2022-08-01
 EVT_0003                 M-Pesa Ethiopia Launch product_launch       2023-08-01
 EVT_0004       Fayda Digital ID Program Rollout infrastructure       2024-01-01
 EVT_0005        Foreign Exchange Liberalization         policy       2024-07-29
 EVT_0006    P2P Transaction Count Surpasses ATM      milestone       2024-10-01
 EVT_0007           M-Pesa EthSwitch Integration    partnership       2025-10-27
 EVT_0008 EthioPay Instant Payment System Launch infrastructure       2025-12-18
 EVT_0009                NFIS-II Strategy Launch         policy       2021-09-01
 EVT_0010      Safaricom Ethiopia Price Increase        pricing       2025-12-15


In [5]:
impact_joined = impact.merge(
    events.rename(columns={'record_id':'parent_id','indicator':'event_name'}),
    on='parent_id', how='left'
)
print(impact_joined[['parent_id','event_name','pillar','related_indicator','impact_direction','impact_magnitude','lag_months']].to_string(index=False))

parent_id                             event_name        pillar  related_indicator impact_direction impact_magnitude  lag_months
 EVT_0001                        Telebirr Launch        ACCESS      ACC_OWNERSHIP         increase             high          12
 EVT_0001                        Telebirr Launch         USAGE USG_TELEBIRR_USERS         increase             high           3
 EVT_0001                        Telebirr Launch         USAGE      USG_P2P_COUNT         increase             high           6
 EVT_0002   Safaricom Ethiopia Commercial Launch        ACCESS         ACC_4G_COV         increase           medium          12
 EVT_0002   Safaricom Ethiopia Commercial Launch AFFORDABILITY    AFF_DATA_INCOME         decrease           medium          12
 EVT_0003                 M-Pesa Ethiopia Launch         USAGE    USG_MPESA_USERS         increase             high           3
 EVT_0003                 M-Pesa Ethiopia Launch        ACCESS     ACC_MM_ACCOUNT         increase      

In [6]:
from datetime import date

new_records = pd.DataFrame([
    # New observation: latest Fayda enrollment figure
    {
        'record_id': 'REC_0034', 'record_type': 'observation', 'category': None,
        'pillar': 'ACCESS', 'indicator': 'Fayda Digital ID Enrollment',
        'indicator_code': 'ACC_FAYDA', 'indicator_direction': 'higher_better',
        'value_numeric': 30000000, 'value_type': 'count', 'unit': 'people',
        'observation_date': '2026-03-05', 'source_name': 'Integrated Biometrics / Biometric Update',
        'source_type': 'news', 'source_url': 'https://www.biometricupdate.com/202603/integrated-biometrics-fingerprint-scanners-facilitate-digital-id-for-inclusion-in-ethiopia',
        'confidence': 'medium',
        'collected_by': 'Mahi', 'collection_date': str(date.today()),
        'original_text': 'More than 30 million people have so far been registered for the ID initiative',
        'notes': 'Extends ACC_FAYDA series beyond the 15M (May 2025) datapoint; shows enrollment nearly doubled in ~10 months.'
    },
    # New event: Fayda mandatory-for-banking regulation
    {
        'record_id': 'EVT_0011', 'record_type': 'event', 'category': 'regulation',
        'pillar': None, 'indicator': 'Fayda Mandatory for Banking Directive',
        'observation_date': '2025-01-08',
        'source_name': 'ID Tech Wire', 'source_type': 'regulator',
        'source_url': 'https://idtechwire.com/ethiopia-mandates-national-digital-id-fayda-for-all-banking-transactions-by-2026/',
        'confidence': 'high',
        'collected_by': 'Mahi', 'collection_date': str(date.today()),
        'original_text': 'All bank account holders must link their accounts to their Fayda IDs by December 31, 2026',
        'notes': 'Not previously captured as a distinct event from the Fayda rollout (EVT_0004); this is a regulatory mandate, not the infrastructure launch itself, and is the first "regulation" category event in the dataset.'
    }
], columns=main.columns)

main_enriched = pd.concat([main, new_records], ignore_index=True)

new_impact_link = pd.DataFrame([
    {
        'record_id': 'IMP_0015', 'parent_id': 'EVT_0011', 'record_type': 'impact_link',
        'category': None, 'pillar': 'ACCESS', 'related_indicator': 'ACC_OWNERSHIP',
        'relationship_type': 'direct', 'impact_direction': 'increase', 'impact_magnitude': 'medium',
        'lag_months': 12, 'evidence_basis': 'documented',
        'comparable_country': None,
        'collected_by': 'Mahi', 'collection_date': str(date.today()),
        'original_text': None,
        'notes': 'Cooperative Bank of Oromia now opens accounts via Fayda eKYC without physical documents, lowering the KYC barrier that has historically excluded undocumented adults from formal accounts.'
    }
], columns=impact.columns)

impact_enriched = pd.concat([impact, new_impact_link], ignore_index=True)

print(main_enriched.tail(3)[['record_id','record_type','category','indicator','observation_date']])
print(impact_enriched.tail(1)[['record_id','parent_id','pillar','related_indicator','impact_direction']])

   record_id  record_type    category                              indicator  \
42  EVT_0010        event     pricing      Safaricom Ethiopia Price Increase   
43  REC_0034  observation        None            Fayda Digital ID Enrollment   
44  EVT_0011        event  regulation  Fayda Mandatory for Banking Directive   

       observation_date  
42  2025-12-15 00:00:00  
43           2026-03-05  
44           2025-01-08  
   record_id parent_id  pillar related_indicator impact_direction
14  IMP_0015  EVT_0011  ACCESS     ACC_OWNERSHIP         increase


C:\Users\bamla\AppData\Local\Temp\ipykernel_12048\3134566842.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  main_enriched = pd.concat([main, new_records], ignore_index=True)
C:\Users\bamla\AppData\Local\Temp\ipykernel_12048\3134566842.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  impact_enriched = pd.concat([impact, new_impact_link], ignore_index=True)


In [7]:
import os
os.makedirs('../data/processed', exist_ok=True)

with pd.ExcelWriter('../data/processed/ethiopia_fi_enriched.xlsx') as writer:
    main_enriched.to_excel(writer, sheet_name='ethiopia_fi_unified_data', index=False)
    impact_enriched.to_excel(writer, sheet_name='Impact_sheet', index=False)

main_enriched.to_csv('../data/processed/ethiopia_fi_enriched_main.csv', index=False)
impact_enriched.to_csv('../data/processed/ethiopia_fi_enriched_impact.csv', index=False)

print("Saved. Files in data/processed:")
print(os.listdir('../data/processed'))

Saved. Files in data/processed:
['ethiopia_fi_enriched.xlsx', 'ethiopia_fi_enriched_impact.csv', 'ethiopia_fi_enriched_main.csv']


In [8]:
log_content = """# Data Enrichment Log

## Task 1: Data Exploration and Enrichment

### Additions

| Record ID | Type | Description | Source | Confidence | Date Added |
|-----------|------|--------------|--------|------------|-------------|
| REC_0034 | observation | Fayda Digital ID enrollment reached 30M (Mar 2026), up from 15M (May 2025) | [Integrated Biometrics / Biometric Update](https://www.biometricupdate.com/202603/integrated-biometrics-fingerprint-scanners-facilitate-digital-id-for-inclusion-in-ethiopia) | medium | {today} |
| EVT_0011 | event (regulation) | Fayda Mandatory for Banking Directive: all bank account holders must link Fayda ID by Dec 31, 2026 | [ID Tech Wire](https://idtechwire.com/ethiopia-mandates-national-digital-id-fayda-for-all-banking-transactions-by-2026/) | high | {today} |
| IMP_0015 | impact_link | EVT_0011 → ACC_OWNERSHIP (ACCESS), increase, medium magnitude, 12-month lag | Documented: Cooperative Bank of Oromia opens accounts via Fayda eKYC without physical documents | medium | {today} |

### Rationale

- **REC_0034** extends the ACC_FAYDA time series with the most recent enrollment figure available, showing enrollment nearly doubled in ~10 months (15M → 30M), which is directly relevant to forecasting ACCESS given Fayda's role as a KYC enabler.
- **EVT_0011** was added as a distinct event from EVT_0004 (Fayda Digital ID Program Rollout, Jan 2024) because it captures a *regulatory mandate* (forcing existing account holders to link IDs) rather than the *infrastructure launch* itself — these are different mechanisms with different expected effects on ACCESS.
- **IMP_0015** models the plausible channel through which this regulation could raise ACC_OWNERSHIP: easier KYC-based account opening at partner banks lowers a historical barrier to account ownership for undocumented adults.

### Considered but not added

- Latest Telebirr user count (54.84M, Jun 2025) and 4G coverage (70.8%, Jun 2025) were checked against the existing dataset and found **already present** (REC_0021, REC_0010) — not duplicated.

### Data Quality Notes

- New observation confidence set to `medium` (news source reporting a program figure, not an official audited report).
- New event confidence set to `high` (multiple corroborating regulatory/industry sources on the mandate and deadline).
""".format(today="{today}")

from datetime import date
log_content = log_content.replace("{today}", str(date.today()))

with open('../data_enrichment_log.md', 'w', encoding='utf-8') as f:
    f.write(log_content)

print("Log written.")

Log written.
